# Notebook 15 — Synthetic Data, Filtering, and Distillation

High-quality synthetic data can amplify a small seed set into a useful training corpus. The point is not to produce random extra text. The point is to generate targeted instruction and response pairs that preserve domain facts, pass filters, and help a smaller or cheaper student specialize.

## What you will build

- a small source-fact library and seed task set
- a synthetic instruction generator that expands requests by persona, channel, and difficulty
- a teacher answer builder with an optional live local-teacher path
- filtering and deduplication checks before training
- a short student distillation run on Colab-class hardware
- an evaluation loop for specialization quality and deployment trade-offs

## Design principle

Synthetic data is useful when it is targeted, filtered, and measured. It is dangerous when it becomes an uninspected pile of self-generated text. The workflow here keeps the pipeline transparent enough to audit inside one notebook.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
from collections import Counter, defaultdict
import hashlib

random.seed(15)

ARTIFACT_DIR = Path("artifacts") / "notebook_15_synthetic_data_and_distillation"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def normalize_text(text):
    return " ".join(text.lower().strip().split())

def stable_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()[:12]

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Define source facts and seed tasks

The teacher should be grounded in something concrete. We start from a small open set of support and operations facts so we can check whether synthetic answers actually preserve useful content.

In [ ]:
source_snippets = [
    {"category": "identity", "fact": "Password reset incidents should verify recent login anomalies before forcing a global credential rotation."},
    {"category": "identity", "fact": "If the reset email queue is delayed, update the status page before asking users to retry repeatedly."},
    {"category": "identity", "fact": "Escalate to identity engineering when MFA failures correlate with a recent certificate or clock-skew change."},
    {"category": "network", "fact": "Retry storms with low CPU usually indicate stuck consumers, poison messages, or a bad upstream timeout policy."},
    {"category": "network", "fact": "A canary rollback is safer than global rollback when only one shard shows elevated packet loss."},
    {"category": "network", "fact": "Do not scale blindly during a queue backlog until you inspect whether one tenant or region is dominating traffic."},
    {"category": "billing", "fact": "Invoice mismatches should confirm tax-region metadata and duplicate jobs before refund approval."},
    {"category": "billing", "fact": "A billing incident summary should name customer impact, backlog size, and the safest manual workaround."},
    {"category": "billing", "fact": "Refund escalation needs an owner, timeline, and evidence that the correction will not re-trigger the faulty job."},
]

seed_tasks = [
    {"seed_id": "identity_triage", "category": "identity", "user_goal": "triage a password reset incident"},
    {"seed_id": "status_update", "category": "identity", "user_goal": "draft a user-facing incident update"},
    {"seed_id": "retry_backlog", "category": "network", "user_goal": "explain a retry storm and next steps"},
    {"seed_id": "rollout_decision", "category": "network", "user_goal": "decide between canary rollback and global rollback"},
    {"seed_id": "refund_plan", "category": "billing", "user_goal": "write a refund escalation plan"},
    {"seed_id": "impact_summary", "category": "billing", "user_goal": "summarize a customer billing incident"},
]

display(pd.DataFrame(source_snippets))
display(pd.DataFrame(seed_tasks))

## Step 2: Expand synthetic user requests

One seed task can become many training examples if you vary persona, format, urgency, and channel while keeping the core task aligned with the same source facts.

In [ ]:
personas = [
    "support lead",
    "new on-call engineer",
    "customer success manager",
]

channels = [
    "chat reply",
    "incident summary",
    "checklist",
]

difficulty_tags = [
    "simple",
    "needs trade-off explanation",
    "high urgency with customer impact",
]

request_rows = []
for seed in seed_tasks:
    for persona in personas:
        for channel in channels:
            for difficulty in difficulty_tags:
                request_rows.append(
                    {
                        "request_id": stable_hash(f'{seed["seed_id"]}|{persona}|{channel}|{difficulty}'),
                        "seed_id": seed["seed_id"],
                        "category": seed["category"],
                        "persona": persona,
                        "channel": channel,
                        "difficulty": difficulty,
                        "prompt": f'As a {persona}, {seed["user_goal"]}. Format the answer as a {channel}. Make it {difficulty}.',
                    }
                )

request_df = pd.DataFrame(request_rows)
print("Synthetic request candidates:", len(request_df))
display(request_df.head(10))

## Step 3: Create a deterministic teacher

For reproducible teaching we build a transparent teacher first. In a production pipeline you might replace this with a larger open model, a stronger prompt template, or a rubric-guided generation stack.

In [ ]:
facts_by_category = defaultdict(list)
for item in source_snippets:
    facts_by_category[item["category"]].append(item["fact"])

def teacher_answer(example):
    facts = facts_by_category[example["category"]]
    summary = facts[0]
    support_fact = facts[1]
    escalation_fact = facts[2]
    return "\n".join(
        [
            f'Summary: {summary}',
            f'Checklist: 1) Confirm scope. 2) Check evidence tied to {support_fact.lower()} 3) Record the owner and next update time.',
            f'Escalate when: {escalation_fact}',
            f'Tone note: respond like a {example["persona"]} writing a {example["channel"]}.',
        ]
    )

synthetic_pairs = []
for row in request_rows:
    answer = teacher_answer(row)
    synthetic_pairs.append({**row, "answer": answer, "teacher_type": "deterministic"})

synthetic_df = pd.DataFrame(synthetic_pairs)
display(synthetic_df.head(6))

## Step 4: Optional live local-teacher augmentation

The notebook already works with the deterministic teacher. If you want extra variety, enable the local-teacher path below. It uses the open model loaded by setup and keeps generation inside Colab.

In [ ]:
USE_LIVE_TEACHER = False
LIVE_TEACHER_LIMIT = 6

def live_teacher_answer(prompt):
    FastLanguageModel.for_inference(model)
    device = next(model.parameters()).device
    messages = [
        {"role": "system", "content": "You are a senior support coach. Produce concise but factual answers with Summary, Checklist, and Escalate when sections."},
        {"role": "user", "content": prompt},
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0,
    )
    answer_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

if USE_LIVE_TEACHER:
    for index in range(min(LIVE_TEACHER_LIMIT, len(synthetic_pairs))):
        synthetic_pairs[index]["answer"] = live_teacher_answer(synthetic_pairs[index]["prompt"])
        synthetic_pairs[index]["teacher_type"] = "live_local_teacher"

synthetic_df = pd.DataFrame(synthetic_pairs)
synthetic_df["teacher_type"].value_counts()

## Step 5: Filter low-quality generations

Filtering matters more than raw volume. We reject examples with weak structure, placeholder language, poor fact coverage, or suspiciously short answers.

In [ ]:
banned_phrases = [
    "as an ai language model",
    "lorem ipsum",
    "todo",
    "placeholder",
]

category_keywords = {
    "identity": ["password", "reset", "mfa", "status", "login"],
    "network": ["retry", "queue", "canary", "rollback", "tenant"],
    "billing": ["invoice", "refund", "billing", "customer", "backlog"],
}

def filter_record(row):
    answer = row["answer"]
    lowered = answer.lower()
    word_count = len(answer.split())
    structure_ok = all(section in answer for section in ["Summary:", "Checklist:", "Escalate when:"])
    no_banned_phrase = not any(bad in lowered for bad in banned_phrases)
    keyword_hits = sum(1 for keyword in category_keywords[row["category"]] if keyword in lowered)
    return {
        "word_count": word_count,
        "structure_ok": structure_ok,
        "no_banned_phrase": no_banned_phrase,
        "keyword_hits": keyword_hits,
        "keep": structure_ok and no_banned_phrase and word_count >= 35 and keyword_hits >= 2,
    }

filter_rows = []
for row in synthetic_pairs:
    checks = filter_record(row)
    filter_rows.append({**row, **checks})

filtered_df = pd.DataFrame(filter_rows)
display(filtered_df[["request_id", "category", "teacher_type", "word_count", "keyword_hits", "keep"]].head(12))
print(filtered_df["keep"].value_counts())

## Step 6: Deduplicate and preserve diversity

Generated datasets often contain near-duplicates. We keep one copy of any normalized prompt plus answer pair and track how much diversity remains by seed task and category.

In [ ]:
kept_df = filtered_df[filtered_df["keep"]].copy()
kept_df["dedupe_key"] = kept_df.apply(
    lambda row: stable_hash(row["prompt"] + " || " + row["answer"]),
    axis=1,
)
kept_df = kept_df.drop_duplicates(subset=["dedupe_key"]).copy()

diversity_summary = (
    kept_df.groupby(["category", "seed_id"])
    .size()
    .reset_index(name="examples")
    .sort_values(["category", "examples"], ascending=[True, False])
)

display(diversity_summary)
print("Kept unique examples:", len(kept_df))

kept_df.to_csv(ARTIFACT_DIR / "synthetic_pairs_filtered.csv", index=False)

## Step 7: Measure dataset amplification

Amplification is the ratio between final kept examples and original seed tasks. Large ratios are only good if the data still looks clean after filtering.

In [ ]:
seed_count = len(seed_tasks)
final_count = len(kept_df)
amplification_factor = round(final_count / seed_count, 2)

amplification_by_category = (
    kept_df.groupby("category")
    .size()
    .reset_index(name="examples")
    .sort_values("examples", ascending=False)
)
amplification_by_category["per_seed"] = amplification_by_category["examples"] / amplification_by_category["category"].map(
    Counter(task["category"] for task in seed_tasks)
)

print("Overall amplification factor:", amplification_factor)
display(amplification_by_category)

ax = amplification_by_category.plot(
    x="category",
    y="examples",
    kind="bar",
    figsize=(7, 4),
    title="Kept synthetic examples by category",
    legend=False,
)
ax.set_ylabel("Examples")
plt.show()

## Step 8: Build the student training set

Distillation means teaching the student to reproduce teacher behavior. Here we package each prompt and answer pair with the model's chat template so the student learns the target style directly.

In [ ]:
SYSTEM_PROMPT = "You are a precise support specialist. Keep the answer factual, structured, and action-oriented."

def render_chat_text(prompt, answer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

student_rows = [
    {
        "text": render_chat_text(row["prompt"], row["answer"]),
        "prompt": row["prompt"],
        "answer": row["answer"],
        "category": row["category"],
    }
    for row in kept_df.to_dict(orient="records")
]

student_dataset = Dataset.from_list(student_rows)
student_split = student_dataset.train_test_split(test_size=0.18, seed=15)

print("Student train rows:", len(student_split["train"]))
print("Student eval rows:", len(student_split["test"]))

## Step 9: Define held-out specialization probes

The held-out set should test style and content without exactly repeating the synthetic prompts that created the dataset.

In [ ]:
eval_cases = [
    {
        "prompt": "Write a checklist for a password reset incident where MFA failures started after a certificate change.",
        "required_terms": ["mfa", "certificate", "checklist", "escalate"],
    },
    {
        "prompt": "Explain why retry storms with low CPU should not trigger blind autoscaling.",
        "required_terms": ["retry", "cpu", "autoscaling", "queue"],
    },
    {
        "prompt": "Draft a short billing incident summary that includes customer impact and backlog size.",
        "required_terms": ["customer", "impact", "backlog", "summary"],
    },
]

display(pd.DataFrame(eval_cases))

## Step 10: Baseline student evaluation

We score the current model with a tiny rubric: did the answer use the right structure and include required domain terms? This is not a full benchmark, but it is enough to compare before and after the distillation slice.

In [ ]:
def generate_student_answer(prompt, max_new_tokens=120):
    FastLanguageModel.for_inference(model)
    device = next(model.parameters()).device
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
    )
    answer_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

def rubric_score(answer, required_terms):
    lowered = answer.lower()
    structure = 1.0 if "summary" in lowered or "checklist" in lowered else 0.0
    keyword_score = sum(1 for term in required_terms if term in lowered) / len(required_terms)
    return round(0.4 * structure + 0.6 * keyword_score, 3)

baseline_eval_rows = []
for case in eval_cases:
    answer = generate_student_answer(case["prompt"])
    baseline_eval_rows.append(
        {
            "prompt": case["prompt"],
            "answer_before": answer,
            "score_before": rubric_score(answer, case["required_terms"]),
        }
    )

baseline_eval_df = pd.DataFrame(baseline_eval_rows)
display(baseline_eval_df)

## Step 11: Configure a short student distillation run

This run is intentionally small so it remains Colab-realistic. In a larger pipeline you would likely distill into a smaller base model and run stronger evals after each checkpoint.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "distilled_student"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=4,
    max_steps=24,
    logging_steps=4,
    evaluation_strategy="steps",
    eval_steps=6,
    save_strategy="steps",
    save_steps=12,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=15,
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=student_split["train"],
    eval_dataset=student_split["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=training_args,
)

print("Trainer ready for distillation.")

## Step 12: Train the student adapter

Think of this as a specialization pass. The adapter is learning the teacher's preferred response pattern in a narrow domain without requiring a massive labeled dataset.

In [ ]:
train_result = trainer.train()
trainer.save_model()

distill_log = pd.DataFrame(trainer.state.log_history)
distill_log.to_csv(ARTIFACT_DIR / "distillation_log.csv", index=False)

display(distill_log.tail(8))

## Step 13: Evaluate the distilled student

The main question is whether the student now answers more like the teacher on held-out tasks. We compare rubric scores and inspect the generated text directly.

In [ ]:
after_eval_rows = []
for case in eval_cases:
    answer = generate_student_answer(case["prompt"])
    after_eval_rows.append(
        {
            "prompt": case["prompt"],
            "answer_after": answer,
            "score_after": rubric_score(answer, case["required_terms"]),
        }
    )

after_eval_df = pd.DataFrame(after_eval_rows)
comparison_df = baseline_eval_df.merge(after_eval_df, on="prompt")
comparison_df["delta"] = comparison_df["score_after"] - comparison_df["score_before"]

display(comparison_df)

ax = comparison_df.set_index("prompt")[["score_before", "score_after"]].plot(
    kind="bar",
    figsize=(9, 4),
    title="Held-out specialization score before vs after distillation",
)
ax.set_ylabel("Rubric score")
plt.show()

## Step 14: Decide how synthetic data supports small-model specialization

The strongest use case for synthetic data is often not making a large model slightly better. It is transferring a stronger teacher's behavior into a smaller, cheaper deployment target.

In [ ]:
deployment_profiles = pd.DataFrame(
    [
        {
            "student_model": "3B instruct student",
            "relative_latency": 1.0,
            "relative_cost": 1.0,
            "best_use": "tight latency and cost budgets",
            "synthetic_data_value": "very high",
        },
        {
            "student_model": "7B instruct student",
            "relative_latency": 1.8,
            "relative_cost": 2.2,
            "best_use": "balanced quality and cost",
            "synthetic_data_value": "high",
        },
        {
            "student_model": "14B teacher or reviewer",
            "relative_latency": 4.0,
            "relative_cost": 5.5,
            "best_use": "teacher generation, filtering, evaluation",
            "synthetic_data_value": "acts as source model",
        },
    ]
)

display(deployment_profiles)

print("Recommended pattern:")
print("- use the larger model for teacher generation and review")
print("- use filtering to keep only clean synthetic pairs")
print("- distill into the smaller student that matches deployment constraints")

## Key takeaways

- Synthetic generation is a multiplier on seed data, not a substitute for filtering.
- Distillation works best when the teacher is grounded in source facts or strong rubrics.
- Dataset amplification should be measured after cleaning, not before.
- Small-model specialization is often the economic reason to build this pipeline.
- Keep a held-out set so you can detect whether the student learned the right behavior instead of memorizing prompts.